In [1]:
import tarfile
import zipfile
import io
import os
import time
import math
import pickle
import itertools as itr
import functools as ft
import regex as re
import chardet
from tqdm.auto import tqdm
from pytictoc import TicToc
import json

import pandas as pd
import numpy as np

import gcsfs
fs = gcsfs.GCSFileSystem()

PROJECT_ID = "arxiv-development"

from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode, LatexGroupNode, LatexMacroNode, LatexCharsNode
from pylatexenc.latex2text import LatexNodes2Text


In [2]:
os.chdir("/home/jupyter/metadata-vertexai/")  # this needs to be the folder where notebook lives
import importlib
import phase_one

In [3]:
from IPython.core.interactiveshell import InteractiveShell
# pretty print all cell's output and not just the last one
InteractiveShell.ast_node_interactivity = "all"

In [4]:
def safe_divide(num, denom):
    return num / denom if denom != 0 else 0.0

Note that id lists were prepared previously from the DB, using:  

```
select concat(paper_id,"v",version) as arx_id from arXiv_metadata
where paper_id LIKE "23%";
and is_withdrawn != 1
and is_current = 1;
```

In [5]:
ror_gspath = 'gs://institutional-extract-scratch/reference/v1.63-2025-04-03-ror-data_schema_v2.json'
fs = gcsfs.GCSFileSystem()
with fs.open(ror_gspath, "r", encoding="utf-8") as f:
    ror_data = json.load(f)
ror_dict = {x['id'].rsplit('/')[-1]: x for x in ror_data}

In [6]:
#ror_dict['043mz5j54']

In [7]:
def get_children_for_ror(target_id):
    res_list = []
    rels = ror_dict[target_id].get('relationships',[])
    for rel in rels:
        rel_type = rel.get('type','')
        if rel_type != 'child':
            continue
        child_id = rel.get('id','').rsplit('/')[-1]
        if child_id:
            res_list.append(child_id)
    return res_list


In [8]:
test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv")
test_ids_df.head()
ids_2311_all = test_ids_df["arx_id"].unique()

,arx_id
0,2311.00001v1
1,2311.00002v1
2,2311.00003v4
3,2311.00004v3
4,2311.00005v1


In [9]:
test_ids_df.shape

(18774, 1)

In [10]:
vip_df = pd.read_csv("gs://institutional-extract-scratch/reference/dashboard_institutions2023_2025-01-21.csv", dtype=str)
vip_df = vip_df[vip_df['is_consortium']=='0']
vip_df.loc[vip_df["orgId"] == '60003088',"ror_id"] = 'https://ror.org/00y4zzh67'  # The GWU
vip_df.loc[vip_df["orgId"] == '60009254',"ror_id"] = 'https://ror.org/02dqehb95'  # 02dqehb95, Purdue University West Lafayette
# University of Colorado, Boulder, 02ttsq026
vip_df.shape
vip_df.head()

(345, 20)

,sid,name,country,country_code,consortia_code,member_type,ror_id,is_consortium,label,comment,is_active,Institution,salsaId,orgId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
0,447,Aalto University,Finland,FI,FinELib,member,https://ror.org/020hwjq30,0,Aalto University,NaN,1,Aalto University,52318446,60103653,NaN,antti.m.rousi@aalto.fi,Antti,Rousi,"Specialist, Research Services",NaN
1,465,Abo Akademi University,Finland,FI,FinELib,member,https://ror.org/029pk6x14,0,Abo Akademi University,NaN,1,Abo Akademi University,NaN,60015375,No contact for school this is the consortium c...,timo.vilen@helsinki.fi,Timo,Vilén,Information Specialist,NaN
2,482,Ames Laboratory,United States,US,NaN,member,https://ror.org/041m9xr71,0,Ames Laboratory,NaN,1,Ames Laboratory,52320619,60008023,NaN,lgraves@iastate.edu,Laura,Graves,NaN,NaN
3,483,Argonne National Lab,United States,US,NaN,member,https://ror.org/05gvnxz63,0,Argonne National Lab,NaN,1,Argonne National Lab,1714199,60028609,NaN,mstraka@anl.gov,Mary,Straka,NaN,NaN
4,16,Australian National University,Australia,AU,CAUL,member,https://ror.org/019wvm592,0,Australian National University,NaN,1,Australian National University,2090464,60008950,NaN,electronic.coordinator@anu.edu.au,NaN,NaN,NaN,NaN


In [11]:
vip_df.head()['ror_id'].apply(lambda x: x.rsplit('/')[-1])

0    020hwjq30
1    029pk6x14
2    041m9xr71
3    05gvnxz63
4    019wvm592
Name: ror_id, dtype: object

In [12]:
ror_df = pd.DataFrame({
    'name':vip_df['name'],
    'ror':vip_df['ror_id'].apply(lambda x: pd.NA if pd.isna(x) else x.rsplit('/')[-1])
})
ror_df = ror_df.dropna()
ror_df.shape
ror_df.head()

(339, 2)

,name,ror
0,Aalto University,020hwjq30
1,Abo Akademi University,029pk6x14
2,Ames Laboratory,041m9xr71
3,Argonne National Lab,05gvnxz63
4,Australian National University,019wvm592


### Eval results

In [13]:
ror_map_df = pd.read_csv("gs://institutional-extract-scratch/reference/matched_results_ror_api.csv", dtype=str)
ror_map_df = ror_map_df.set_index('Primary Org Id')
ror_map_df.head()
vip_df = vip_df.set_index('orgId')
ror_map_df = pd.merge(
    ror_map_df, 
    vip_df[['ror_id']],
    how='left', left_index=True, right_index=True,
)
ror_map_df['ror_id'] = ror_map_df['ror_id'].fillna(ror_map_df['ROR ID'])
ror_map_df.drop(columns=['ROR ID'], inplace=True)
ror_map_df['ror'] = ror_map_df['ror_id'].apply(lambda x: pd.NA if pd.isna(x) else x.rsplit('/')[-1])
rors_in_map = set(ror_map_df['ror'].unique())
ror_map_df = ror_map_df.reset_index()
ror_map_df.shape
ror_map_df.head()

,Primary Org Name,Country Name,ROR ID
Primary Org Id,,,
60000009,Villanova University,United States,https://ror.org/02g7kd627
60000011,Saitama Institute of Technology,Japan,https://ror.org/01pkeax38
60000015,Lusófona University,Portugal,https://ror.org/05xxfer42
60000021,Atatürk Üniversitesi,Turkey,https://ror.org/03je5c526
60000027,KLA Corporation,United States,https://ror.org/04zdyxh40


(13368, 5)

,Primary Org Id,Primary Org Name,Country Name,ror_id,ror
0,60000009,Villanova University,United States,https://ror.org/02g7kd627,02g7kd627
1,60000011,Saitama Institute of Technology,Japan,https://ror.org/01pkeax38,01pkeax38
2,60000015,Lusófona University,Portugal,https://ror.org/05xxfer42,05xxfer42
3,60000021,Atatürk Üniversitesi,Turkey,https://ror.org/03je5c526,03je5c526
4,60000027,KLA Corporation,United States,https://ror.org/04zdyxh40,04zdyxh40


In [14]:
ror_map_df.index

RangeIndex(start=0, stop=13368, step=1)

In [15]:
vip_df[vip_df['name'].str.contains("George Washington")]
ror_map_df[ror_map_df['Primary Org Name'].str.contains("George Washington")]

,sid,name,country,country_code,consortia_code,member_type,ror_id,is_consortium,label,comment,is_active,Institution,salsaId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
orgId,,,,,,,,,,,,,,,,,,,
60003088,610,The George Washington University,NaN,NaN,NaN,member,https://ror.org/00y4zzh67,0,The George Washington University,NaN,1,The George Washington University,NaN,NaN,jel@gwu.edu,NaN,NaN,NaN,NaN


,Primary Org Id,Primary Org Name,Country Name,ror_id,ror
613,60003088,The George Washington University,United States,https://ror.org/00y4zzh67,00y4zzh67


In [16]:
scopus_df = pd.read_csv("gs://institutional-extract-scratch/training/2311_scopus_17416.csv.zip", dtype=str)
scopus_all = set(scopus_df["ArXiv Id"].unique())
scopus_positive = scopus_df[scopus_df['Primary Org Id'] == 60027550]["ArXiv Id"].unique()

scopus_df = pd.merge(
    scopus_df, 
    ror_map_df[["Primary Org Id", "ror"]], 
    how='left',
    on='Primary Org Id'
)
scopus_all = set(scopus_df["paper_id"].unique())

In [17]:
scopus_df['paper_id'].nunique()
scopus_df.shape
scopus_df.head()

17416

(80750, 9)

,Primary Org Id,Primary Org Name,Primary Org City,Primary Org State,Primary Org Country,ArXiv Id,Affiliation Sequence Number,paper_id,ror
0,60006297,University of Pennsylvania,Philadelphia,PA,United States,2311.03477v1,1,2311.03477,00b30xv10
1,60024190,"Institute of Plasma Physics, Academy of Scienc...",Prague,NaN,Czech Republic,2311.04187v1,9,2311.04187,01h494015
2,60025641,Universität Freiburg,Freiburg im Breisgau,Baden-Wurttemberg,Germany,2311.04557v1,1,2311.04557,0245cg223
3,60114755,"Istituto Nazionale di Fisica Nucleare, Sezione...",Milan,NaN,Italy,2311.14088v1,27,2311.14088,04w4m6z96
4,60115855,Trento Institute for Fundamental Physics and A...,Povo,TN,Italy,2311.09750v1,2,2311.09750,00nhs3j29


In [18]:
res_df = pd.read_csv("gs://institutional-extract-scratch/output/2311_db_all.csv.zip")
res_df['paper_id'] = res_df['arx_id'].apply(lambda x: x.rsplit('v',1)[0])
res_df.shape
res_df.head()

(48340, 4)

,arx_id,name,ror,paper_id
0,2311.00920v2,"University of Science and Technology of China,...",04c4dkn09,2311.00920
1,2311.16784v1,"University of Leeds, Leeds",024mrxd33,2311.16784
2,2311.16083v1,"University of Strathclyde, Glasgow",00n3w3b69,2311.16083
3,2311.16083v1,"University of Leeds, Leeds",024mrxd33,2311.16083
4,2311.18524v1,"ETH Zurich, Zurich",05a28rw58,2311.18524


In [19]:
skip_inst = {
    '01nsd7y51': 'CSIC - Geociencias Barcelona (GEO3BCN)',
    '05dsysc59': 'CSIC - Instituto de Carboquímica (ICB)',
    '04zdays56': 'CSIC - Instituto de Biologia Molecular y Celul...',
    '03hasqf61': 'CSIC - Institut de Ciència de Materials de Bar...',
    '02h7vfp25': 'CSIC - Instituto de Cerámica y Vidrio (ICV)',
    '04qayn356': 'CSIC - Instituto de Ciencias Marinas de Andalu...',
}

scopus_to_rerun = set()
vip_results = []
for ror in tqdm(ror_df['ror'].unique()):
    if ror in skip_inst:
        continue
    if not ror in rors_in_map:
        print(f"'{ror}': {ror_df.loc[ror_df['ror']==ror,'name'].iloc[0]}")
        continue
    inst_name = ror_map_df[ror_map_df['ror']==ror]['Primary Org Name'].iloc[0]
    sub_ids = get_children_for_ror(ror)
    sub_ids.append(ror)
    scopus_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['paper_id'].unique())
    scopus_arx_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['ArXiv Id'].unique())
    scopus_to_rerun = scopus_to_rerun.union(scopus_arx_ids)
    res_ids = set(res_df[res_df['ror'].isin(sub_ids)]['arx_id'].apply(lambda x: x.rsplit('v')[0]).unique())
    res_ids = res_ids.intersection(scopus_all) # only consider cases in the scopus data
    res = {
        "name": inst_name,
        "ror": ror,
        "TP": len(res_ids.intersection(scopus_ids)), 
        "FP": len(res_ids - scopus_ids), 
        "FN": len(scopus_ids - res_ids), 
        "TN": len((scopus_all - scopus_ids) - res_ids)
    }
    res['precision'] = safe_divide(res['TP'], res['TP'] + res['FP'])
    res['recall'] = safe_divide(res['TP'], res['TP'] + res['FN'])
    if (res['precision'] + res['recall']) > 0:
        res['F1'] = safe_divide(2 * res['precision'] * res['recall'], res['precision'] + res['recall'])
    else:
        res['F1'] = 0.0
    vip_results.append(res)


  0%|          | 0/338 [00:00<?, ?it/s]

'03srn9y98': CSIC - Instituto de Química Avanzada de Cataluña (IQAC)
'006gw6z14': CSIC- Estación Biológica de Doñana EBD
'04nrv3s86': CSIC-UMA - Instituto de Hortofruticultura Subtropical y Mediterranea La Mayora (IHSM)
'000nhpy59': CSIC-UMH - Instituto de Neurociencias (IN)
'021f7p178': Lib4RI
'052rrw050': National Astronomical Observatory of Japan
'049bh0z35': National Library of Sweden
'00bwtjf83': Tampere University of Applied Sciences
'028rypz17': Université Paris-Sud


In [20]:
vip_res_df = pd.DataFrame.from_records(vip_results)
vip_res_df.head()

,name,ror,TP,FP,FN,TN,precision,recall,F1
0,Aalto University,020hwjq30,55,0,26,17335,1.000000,0.679012,0.808824
1,Åbo Akademi University,029pk6x14,2,0,0,17414,1.000000,1.000000,1.000000
2,Ames Laboratory,041m9xr71,4,0,2,17410,1.000000,0.666667,0.800000
3,Argonne National Laboratory,05gvnxz63,40,2,17,17357,0.952381,0.701754,0.808081
4,The Australian National University,019wvm592,71,2,7,17336,0.972603,0.910256,0.940397


In [21]:
len(scopus_to_rerun)

9769

In [22]:
vip_res_df.to_csv("vip_results_2025-04-27_improved_eval.csv")

In [23]:
total_gt_cases = vip_res_df[['TP', 'FP', 'FN', 'TN']].iloc[0].sum()

In [219]:
cutoff = 10
tn_min = total_cases - cutoff
vip_res_df.query("TN <= @tn_min").sort_values('F1', ascending=True).head(20)

,name,ror,TP,FP,FN,TN,precision,recall,F1
67,Institute of Mathematical Sciences India,04zp24820,1,9,10,17396,0.100000,0.090909,0.095238
176,The University of British Columbia,03kgj4539,12,0,110,17294,1.000000,0.098361,0.179104
70,Commissariat a l'Energie Atomique et aux Energ...,00jjx8s55,36,3,179,17198,0.923077,0.167442,0.283465
44,CSIC-UAM - Instituto de Física Teórica (IFT),022r8mj40,6,2,27,17381,0.750000,0.181818,0.292683
59,CSIC-UV - Instituto de Física Corpuscular,017xch102,6,0,26,17384,1.000000,0.187500,0.315789
144,Foundation for Fundamental Research on Matter,00f9tz983,10,1,35,17370,0.909091,0.222222,0.357143
119,King's College London,0220mzb33,62,0,199,17155,1.000000,0.237548,0.383901
222,University of Cape Town,03p74gp79,9,0,25,17382,1.000000,0.264706,0.418605
104,Institute of Physics of the Czech Academy of S...,02yhj4v17,10,2,25,17379,0.833333,0.285714,0.425532
274,"University of the Witwatersrand, Johannesburg",03rp50x72,9,2,22,17383,0.818182,0.290323,0.428571


In [24]:
scopus_df.head()

,Primary Org Id,Primary Org Name,Primary Org City,Primary Org State,Primary Org Country,ArXiv Id,Affiliation Sequence Number,paper_id,ror
0,60006297,University of Pennsylvania,Philadelphia,PA,United States,2311.03477v1,1,2311.03477,00b30xv10
1,60024190,"Institute of Plasma Physics, Academy of Scienc...",Prague,NaN,Czech Republic,2311.04187v1,9,2311.04187,01h494015
2,60025641,Universität Freiburg,Freiburg im Breisgau,Baden-Wurttemberg,Germany,2311.04557v1,1,2311.04557,0245cg223
3,60114755,"Istituto Nazionale di Fisica Nucleare, Sezione...",Milan,NaN,Italy,2311.14088v1,27,2311.14088,04w4m6z96
4,60115855,Trento Institute for Fundamental Physics and A...,Povo,TN,Italy,2311.09750v1,2,2311.09750,00nhs3j29


In [31]:
name_txt = "New York University"
scopus_df[scopus_df['Primary Org Name'].str.contains(name_txt)].head()
ror_map_df[ror_map_df['Primary Org Name'].str.contains(name_txt)]
vip_df[vip_df['name'].str.contains(name_txt)]

,Primary Org Id,Primary Org Name,Primary Org City,Primary Org State,Primary Org Country,ArXiv Id,Affiliation Sequence Number,paper_id,ror
214,60021784,New York University,New York,NY,United States,2311.02979v1,6,2311.02979,0190ak572
1271,60021784,New York University,New York,NY,United States,2311.05862v1,3,2311.05862,0190ak572
1272,60021784,New York University,New York,NY,United States,2311.02774v1,1,2311.02774,0190ak572
1278,60021784,New York University,New York,NY,United States,2311.17017v1,1,2311.17017,0190ak572
1283,60021784,New York University,New York,NY,United States,2311.14023v1,2,2311.14023,0190ak572


,Primary Org Id,Primary Org Name,Country Name,ror_id,ror
4124,60021784,New York University,United States,https://ror.org/0190ak572,0190ak572


,sid,name,country,country_code,consortia_code,member_type,ror_id,is_consortium,label,comment,is_active,Institution,salsaId,Comment,Usage Contact Email,First Name,Last Name,Title,Unnamed: 17
orgId,,,,,,,,,,,,,,,,,,,
60021784,4,New York University,United States,US,NaN,member,https://ror.org/0190ak572,0,New York University,NaN,1,New York University,5231194,NaN,bm73@nyu.edu,NaN,NaN,NaN,NaN


In [35]:
ror = "0190ak572"
sub_ids = get_children_for_ror(ror)
sub_ids.append(ror)
print(sub_ids)
scopus_ids = set(scopus_df[scopus_df['ror'].isin(sub_ids)]['paper_id'].unique())
len(scopus_ids)

['037tm7f56', '00e5k0821', '02pthyn77', '05mq03431', '01rz15025', '02vpsdb40', '0190ak572']


173

In [36]:
named_inst = res_df[res_df['paper_id'].isin(scopus_ids)].value_counts('name', ascending=False)
named_inst.head()
res_df[res_df['paper_id'].isin(scopus_ids)].groupby(['name'])['ror'].value_counts(ascending=False).sort_values(ascending=False).head(20)
res_df[res_df['paper_id'].isin(scopus_ids)].head(10)

name
New York University, New York               87
New York University, Abu Dhabi              17
New York University Abu Dhabi, Abu Dhabi    10
New York University,                         9
CERN, Geneva                                 6
Name: count, dtype: int64

name                                                  ror      
New York University, Abu Dhabi                        00e5k0821    17
New York University Abu Dhabi, Abu Dhabi              00e5k0821    10
New York University,                                  0190ak572     9
University of Chicago, Chicago                        024mw5h28     6
CERN, Geneva                                          01ggx4157     6
Flatiron Institute, New York                          00sekdz59     6
Princeton University, Princeton                       00hx57361     5
Columbia University, New York                         00hj8s172     5
NYU,                                                  0190ak572     5
Sorbonne Université, Paris                            02en5vm52     4
New York University, Shanghai                         02vpsdb40     4
New York University                                   02mp2av58     4
NYU Shanghai, Shanghai                                02vpsdb40     4
University of Maryland, Co

,arx_id,name,ror,paper_id
92,2311.03707v1,"Massachusetts Institute of Technology,",042nb2s44,2311.03707
93,2311.03707v1,"Stanford University,",00f54p054,2311.03707
94,2311.03707v1,"New York University,",0190ak572,2311.03707
95,2311.03707v1,Beijing University of Posts and Telecommunicat...,04w9fbh59,2311.03707
96,2311.03707v1,"The Hong Kong Chinese University,",00t33hh48,2311.03707
97,2311.03707v1,The Hong Kong City University,03q8dnn23,2311.03707
870,2311.13480v2,"NYU Shanghai, Shanghai",02vpsdb40,2311.13480
871,2311.13480v2,"Courant Institute of Mathematical Sciences,",037tm7f56,2311.13480
872,2311.13480v2,Beijing Institute of Mathematical Sciences and...,05t6hvr95,2311.13480
890,2311.01587v2,"New York University, Abu Dhabi",00e5k0821,2311.01587


In [200]:
paper_grps = res_df[res_df['paper_id'].isin(scopus_ids)].groupby(['arx_id'])['ror'].unique().reset_index()
paper_grps[paper_grps['ror'].apply(lambda x: (i not in sub_ids for i in x))]

,arx_id,ror
0,2311.00018v1,"[000e0be47, nan, 03vek6s52, 01zkghx44, 00hj8s1..."
1,2311.00208v3,"[05kb8h459, nan, 016fqsx25, 00mkhxb43, 03v76x132]"
2,2311.00258v1,[nan]
5,2311.00647v3,"[052gg0110, 02en5vm52, nan, 04raf6v53, 013meh722]"
6,2311.00748v2,"[05qwgg493, 00sekdz59, nan]"
...,...,...
165,2311.17992v1,"[024mw5h28, nan]"
166,2311.18007v1,"[046sh6j17, nan, 03vek6s52, 00h2vm590]"
168,2311.18245v1,[nan]
169,2311.18298v2,"[01ggx4157, 01swzsf04]"


In [204]:
all(i not in sub_ids for i in paper_grps['ror'][0])

True

In [20]:
arx_id = '2311.15491v1' #'2311.00018v1'
phase_one.get_single_file_results(arx_id, verbose=True)

Processing ftp/arxiv/papers/2311/2311.15491.tar.gz
Processing ftp/arxiv/papers/2311/2311.15491.gz
Failed to read ftp/arxiv/papers/2311/2311.15491.gz with detected encoding {detected_encoding}: {e}
Processing txt/arxiv/2311/2311.15491v1.txt
1. Indian Institute of Technology Hyderabad, Hyderabad



[('2311.15491v1',
  'Indian Institute of Technology Hyderabad, Hyderabad',
  '01j4v3x97')]

In [37]:
target_rors = ['02mp2av58']
res_df[res_df['paper_id'].isin(scopus_ids) & res_df['ror'].isin(target_rors)]

,arx_id,name,ror,paper_id
11587,2311.17969v1,Texas A&M University,02mp2av58,2311.17969
11588,2311.17969v1,New York University,02mp2av58,2311.17969
21437,2311.05877v1,New York University,02mp2av58,2311.05877
22638,2311.08970v1,University of Florida,02mp2av58,2311.08970
24791,2311.03534v2,Microsoft Research,02mp2av58,2311.03534
24792,2311.03534v2,Meta,02mp2av58,2311.03534
36274,2311.18494v1,Yandex LLC,02mp2av58,2311.18494
36276,2311.18494v1,New York University,02mp2av58,2311.18494
44177,2311.16098v1,Meta,02mp2av58,2311.16098
44191,2311.03386v1,New York University,02mp2av58,2311.03386


In [30]:
name_txt = "University of Colorado"
res_df[res_df['paper_id'].isin(scopus_ids) & res_df['name'].str.contains(name_txt)].head()

,arx_id,name,ror,paper_id
1931,2311.01187v1,"University of Colorado Boulder, Boulder",02ttsq026,2311.01187
6729,2311.18020v2,"University of Colorado Boulder, Boulder",02ttsq026,2311.18020
8553,2311.07483v1,"University of Colorado, Boulder",02ttsq026,2311.07483
8755,2311.06424v2,"University of Colorado Boulder,",02ttsq026,2311.06424
14750,2311.13322v2,"University of Colorado Boulder, Boulder",02ttsq026,2311.13322


## Scratch

In [ ]:
ror
(ror in skip_inst)
ror_df[ror_df['ror']==ror]
ror_map_df[ror_map_df['ror']==ror]

In [ ]:
ror_df.loc[ror_df['ror']==ror,'name'].iloc[0]

In [ ]:
res = {
    "ror": ror,
    "TP": len(res_ids.intersection(scopus_ids)), 
    "FP": len(res_ids - scopus_ids), 
    "FN": len(scopus_ids - res_ids), 
    "TN": len((scopus_all - scopus_ids) - res_ids)
}

In [ ]:
list(itr.islice(scopus_ids, 10))

In [ ]:
list(itr.islice(res_ids, 10))

In [ ]:
#%%time
#res_dict = {}
#for arx_id in tqdm(false_positive): #scopus_positive:
#    #print(arx_id)
#    paper_id = arx_id.split("v")[0]
#    res = phase_one.send_one_submission_to_gemini(arx_id)
#    #print(f"\n{paper_id}\n{res}")
#    res_dict[paper_id] = res
#

### Phase 2 Name --> ROR id

Based on FAISS

In [183]:
ror_finder = phase_one.ROR_FINDER

In [207]:
ror_finder5 = phase_one.ROR_FINDER

In [208]:
ror_finder.qa_chain.invoke({"query": 'New York University, New York'})

{'query': 'New York University, New York',
 'result': 'null\n',
 'source_documents': [Document(id='6e603100-4e7c-447c-8e6c-d9f01ab5aa5f', metadata={}, page_content='York College, City University of New York, New York — https://ror.org/015a1ak54'),
  Document(id='a938e6e9-78ca-4711-9b10-485ed79801ff', metadata={}, page_content='City University of New York, New York — https://ror.org/00453a208')]}

In [209]:
ror_finder5.qa_chain.invoke({"query": 'New York University, New York'})

{'query': 'New York University, New York',
 'result': '0190ak572\n',
 'source_documents': [Document(id='6e603100-4e7c-447c-8e6c-d9f01ab5aa5f', metadata={}, page_content='York College, City University of New York, New York — https://ror.org/015a1ak54'),
  Document(id='a938e6e9-78ca-4711-9b10-485ed79801ff', metadata={}, page_content='City University of New York, New York — https://ror.org/00453a208'),
  Document(id='5cdb0189-0b74-46c8-89e7-f40fb3e3b4d0', metadata={}, page_content='New York University, New York — https://ror.org/0190ak572'),
  Document(id='5368175c-fbb5-4dfc-957b-e8840930398c', metadata={}, page_content='York University, York — https://ror.org/022jz8688'),
  Document(id='d07f8630-c036-4e23-8095-4a3fc3ca86d8', metadata={}, page_content='University of York, York — https://ror.org/04m01e293')]}

In [212]:
%%time
ror_finder5.get_ror('New York University, New York')

CPU times: user 8 µs, sys: 1e+03 ns, total: 9 µs
Wall time: 13.4 µs


'0190ak572'

In [ ]:
%%time
ror_finder.get_ror('Universitas Indonesia,')

In [52]:
source_text = '''
\author{
    Sonish Sivarajkumar, MS\textsuperscript{1}\textsuperscript{,2}\thanks{Present address: School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA. Work was done while at Molecular Robotics, Kerala, India.},
    Pratyush Tandale, MS\textsuperscript{3}, 
    Ankit Bhardwaj, BS\textsuperscript{4}, \\
    Kipp W. Johnson, MD,PhD\textsuperscript{5}, 
    Anoop Titus, MD\textsuperscript{6}, 
    Benjamin S. Glicksberg, PhD\textsuperscript{7},\\
    Shameer Khader, PhD, MPH\textsuperscript{8}\textsuperscript{\dag}, 
    Kamlesh K. Yadav, PhD\textsuperscript{9, 10}\textsuperscript{\dag}, \\
    Lakshminarayanan Subramanian, PhD\textsuperscript{4}\thanks{Corresponding authors: shameer.khader20@imperial.ac.uk, kamlesh.yadav@tamu.edu, lakshmi@cs.nyu.edu}
}
'''.strip()

inst_list = '''
1. University of Pittsburgh, Pennsylvania
2. Texas A&M University
3. New York University
'''.strip()

VERIFY_TEMPLATE = """
Match the institution names in the LIST_OF_NAMES with the contents of the SOURCE_TEXT.
Answer "True" if ALL the institutions in the LIST_OF_NAMES are present in the SOURCE_TEXT, otherwise answer "False"\n
Only respond with "True" or "False".
### LIST_OF_NAMES:\n
{inst_list}\n\n

### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()

VERIFY_TEMPLATE = """
Compare the institution names in the LIST_OF_NAMES with the contents of the SOURCE_TEXT.\n
For each instituion name in LIST_OF_NAMES, determine if the SOURCE_TEXT refers the institution.\n 
If the SOURCE_TEXT refers to one of the institutions, identify the substring that refers to the institution.\n

### LIST_OF_NAMES:\n
{inst_list}\n\n

### SOURCE_TEXT:\n
{source_text}\n\n
""".strip()

phase_one.verify_with_gemini_api(inst_list, source_text, template=VERIFY_TEMPLATE)

'Here\'s a comparison of the institutions in `LIST_OF_NAMES` with the `SOURCE_TEXT`:\n\n1. **University of Pittsburgh, Pennsylvania:**  The `SOURCE_TEXT` refers to this institution. The substring that refers to it is "University of Pittsburgh, Pennsylvania".\n\n2. **Texas A&M University:** The `SOURCE_TEXT` refers to this institution indirectly through the email address "kamlesh.yadav@tamu.edu".  While not the full name, "tamu.edu" is a widely recognized abbreviation for Texas A&M University.\n\n3. **New York University:** The `SOURCE_TEXT` refers to this institution indirectly through the email address "lakshmi@cs.nyu.edu". Similar to the previous point, "nyu.edu" is a clear abbreviation for New York University.\n'

In [188]:
phase_one.get_single_file_results('2311.17969v1', verbose=True)


Processing ftp/arxiv/papers/2311/2311.17969.tar.gz
0: \author{
    Sonish Sivarajkumar, MS\textsuperscript{1}\textsuperscript{,2}\thanks{Present address: School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA. Work was done while at Molecular Robotics, Kerala, India.},
    Pratyush Tandale, MS\textsuperscript{3}, 
    Ankit Bhardwaj, BS\textsuperscript{4}, \\
    Kipp W. Johnson, MD,PhD\textsuperscript{5}, 
    Anoop Titus, MD\textsuperscript{6}, 
    Benjamin S. Glicksberg, PhD\textsuperscript{7},\\
    Shameer Khader, PhD, MPH\textsuperscript{8}\textsuperscript{\dag}, 
    Kamlesh K. Yadav, PhD\textsuperscript{9, 10}\textsuperscript{\dag}, \\
    Lakshminarayanan Subramanian, PhD\textsuperscript{4}\thanks{Corresponding authors: shameer.khader20@imperial.ac.uk, kamlesh.yadav@tamu.edu, lakshmi@cs.nyu.edu}
}
,
    Pratyush Tandale, MS
,
    Pratyush Tandale, MS
, 
    Ankit Bhardwaj, BS
, 
, 
    Anoop Titus, MD
, 
    Benjamin S. Glicksberg, PhD
,
, 
    K

[('2311.17969v1', 'University of Pittsburgh, Pennsylvania', '01an3r305'),
 ('2311.17969v1', 'Georgetown University, Washington', '05vzafd60'),
 ('2311.17969v1', 'New York University, New York', '0190ak572'),
 ('2311.17969v1', 'Mount Sinai Health System, New York', '04kfn4587'),
 ('2311.17969v1', 'Houston Methodist, Houston', '027zt9171'),
 ('2311.17969v1', 'Imperial College London, London', '041kmwe10'),
 ('2311.17969v1', 'Texas A&M University, Houston', '01f5ytq51')]

### Clean up results

In [ ]:
res_df = pd.read_csv("gs://institutional-extract-scratch/output/2311_db_all.csv.zip")

In [ ]:
res_df.shape
res_df.head()

In [ ]:
# probably html formatted files
error_files = res_df[res_df['name']=='error']['arx_id'].to_list()
len(error_files)

In [ ]:
no_ror_df = res_df[pd.isna(res_df['name']) | pd.isna(res_df['ror'])]
no_ror_df.shape
no_ror_df.head()

In [ ]:
nn_df = res_df[pd.isna(res_df['name'])]
nn_df.shape
nn_df.head()

In [ ]:
res_list = []
for arx in tqdm(nn_df['arx_id'][:5]):
    res_list.append(phase_one.get_single_file_results(arx))

In [ ]:
res_list

In [18]:
importlib.reload(phase_one)

<module 'phase_one' from '/home/jupyter/metadata-vertexai/phase_one.py'>

### Examine extract

In [181]:
phase_one.get_single_file_results(arx_id, verbose=True)

Processing ftp/arxiv/papers/2311/2311.17969.tar.gz
0: \author{
    Sonish Sivarajkumar, MS\textsuperscript{1}\textsuperscript{,2}\thanks{Present address: School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA. Work was done while at Molecular Robotics, Kerala, India.},
    Pratyush Tandale, MS\textsuperscript{3}, 
    Ankit Bhardwaj, BS\textsuperscript{4}, \\
    Kipp W. Johnson, MD,PhD\textsuperscript{5}, 
    Anoop Titus, MD\textsuperscript{6}, 
    Benjamin S. Glicksberg, PhD\textsuperscript{7},\\
    Shameer Khader, PhD, MPH\textsuperscript{8}\textsuperscript{\dag}, 
    Kamlesh K. Yadav, PhD\textsuperscript{9, 10}\textsuperscript{\dag}, \\
    Lakshminarayanan Subramanian, PhD\textsuperscript{4}\thanks{Corresponding authors: shameer.khader20@imperial.ac.uk, kamlesh.yadav@tamu.edu, lakshmi@cs.nyu.edu}
}
,
    Pratyush Tandale, MS
,
    Pratyush Tandale, MS
, 
    Ankit Bhardwaj, BS
, 
, 
    Anoop Titus, MD
, 
    Benjamin S. Glicksberg, PhD
,
, 
    K

[('2311.17969v1', 'University of Pittsburgh, Pennsylvania', '01an3r305'),
 ('2311.17969v1', 'Georgetown University, Washington DC', '05vzafd60'),
 ('2311.17969v1', 'New York University, New York', 'null'),
 ('2311.17969v1', 'Mount Sinai Health System, New York', '04kfn4587'),
 ('2311.17969v1', 'Houston Methodist, Houston', '027zt9171'),
 ('2311.17969v1', 'Imperial College London, London', '041kmwe10'),
 ('2311.17969v1', 'Texas A&M University, Houston', 'null')]

In [178]:
arx_id = '2311.17969v1'

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
tex_main = phase_one.find_main_tex_source_in_tar(tar_path)[0]

doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""
from google.cloud import storage
PROJECT_ID = "arxiv-development"
PRD_PROJECT = 'arxiv-production'
PRD_BUCKET_LOC = 'arxiv-production-data' 

client = storage.Client(project=PRD_PROJECT)
bucket = client.bucket(PRD_BUCKET_LOC)
blob = bucket.blob(tar_path)
tar_bytes = blob.download_as_bytes()
if tar_path.endswith(".tar.gz"):
    try:
        with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r') as in_tar:
            fp = in_tar.extractfile(tex_main)
            wrapped_file = io.TextIOWrapper(fp, newline=None, encoding='utf-8') #universal newlines
            source_text = phase_one.pre_format(wrapped_file.read())
    except UnicodeDecodeError:
        try:
            with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r') as in_tar:
                fp = in_tar.extractfile(tex_main)
                raw_data = in_tar.extractfile(tex_main).peek(10000)
                result = chardet.detect(raw_data)
                detected_encoding = result["encoding"]
                wrapped_file = io.TextIOWrapper(
                    fp, 
                    newline=None, 
                    encoding=detected_encoding, 
                    errors="replace"
                ) #universal newlines
                source_text = wrapped_file.read()
        except Exception as e:
            print(
                f"Failed to read {tar_path}-{tex_main} with"
                " detected encoding {detected_encoding}: {e}"
            )
            #return None
else:
    try:
        with gzip.open(io.BytesIO(tar_bytes), 'rt', encoding='utf-8') as in_gz:
            source_text = in_gz.read()
    except UnicodeDecodeError:
        try:
            with gzip.open(io.BytesIO(tar_bytes), 'rb') as in_gz:
                raw_data = in_gz.peek(10000)
                result = chardet.detect(raw_data)
                detected_encoding = result["encoding"]
            with gzip.open(
                io.BytesIO(tar_bytes),
                'rt', 
                encoding=detected_encoding
            ) as in_gz:
                source_text = in_gz.read()
        except Exception as e:
            print(
                f"Failed to read {tar_path}-{tex_main} with"
                " detected encoding {detected_encoding}: {e}"
            )

# Remove LaTeX comments (lines starting with non-escaped %)
content = re.sub(r"(?<!\\)%.*", "", source_text)
#res_list = []

# try parsing latex:
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "affiliation", "affil", "affiliations",
    "address",
    "cmsinstitute",
])
supstr = set([
    "\\textsuperscript",
])
latex_extracted_institutions = []
try:
    lxwkr = LatexWalker(content)
    (nodelist, pos, len_) = lxwkr.get_latex_nodes()
    focus_nodes = [
      (i,node) for i,node in enumerate(nodelist)
      if hasattr(node, "macroname") and node.macroname in auth_macros
    ]
    if focus_nodes:
        for i,node in focus_nodes:
            latex_extracted_institutions.append(node.latex_verbatim())
            try:
                follow_node = nodelist[i+1]
                if isinstance(follow_node, LatexGroupNode):
                    latex_extracted_institutions.append(follow_node.latex_verbatim())
            except IndexError:
                pass
        if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
            sup_res = extract_texsuperscript(nodelist)
            latex_extracted_institutions.extend(sup_res)
    else:
        doc = [
            node for node in nodelist
            if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document'
        ]
        if doc:
            focus_doc_nodes = [
              (i,node) for i, node in enumerate(doc[0].nodelist)
              if isinstance(node, LatexMacroNode) and node.macroname in auth_macros
            ]
            for i, node in focus_doc_nodes:
                latex_extracted_institutions.append(node.latex_verbatim())
                try:
                    follow_node = doc[0].nodelist[i+1]
                    if isinstance(follow_node, LatexGroupNode):
                        latex_extracted_institutions.append(follow_node.latex_verbatim())
                except IndexError:
                    pass
            if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
                sup_res = extract_texsuperscript(doc[0].nodelist)
                latex_extracted_institutions.extend(sup_res)
    if latex_extracted_institutions:
        #res_list.append(latex_extracted_institutions)
        #yield "\n".join(latex_extracted_institutions)
        pass
except Exception as e:
    print(f"Overly broad except in extract_pre_abstract_content(): {e}")
    pass

#  "recursive" regex:
#   ((?>[^{}]+|\{(?1)\})*)
# optional brackets
#   (:?\[\d+\])?\s*
# This matches text possibly containing normal characters or nested braces,
# until the outermost braces are matched.
# If your LaTeX does not have deep nesting, this mainly ensures things like $^{1}$ are correctly parsed.
institution_patterns = [
    r"\\affiliation\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\institute\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\address\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\inst\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\affil\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\author\s*(:?\[\d+\])?\s*{[^}]+}{([^}]+)}",
    r"\\cmsinstitute\s*(:?\[\d+\])?\s*{[^}]+}{([^}]+)}",
]

extracted_institutions = []
for pattern in institution_patterns:
    # Use regex.findall with DOTALL to allow '.' to match newlines
    matches = re.findall(pattern, content, flags=re.DOTALL)
    if matches:
        # Strip each match and add to list
        for m in matches:
            if isinstance(m, tuple):
                extracted_institutions.append(" ".join(m_i for m_i in m))
            else:
                extracted_institutions.extend(m.strip() for m in matches if m.strip())

# If any institution info is extracted, return the deduplicated joined text
if extracted_institutions:
    # You can change the join method; here we join by newline and use set to deduplicate
    #return "\n".join(set(extracted_institutions))
    #res_list.append("\n".join(set(extracted_institutions)))
    #yield "\n".join(set(extracted_institutions))
    pass

# If no institution found, try extracting the text before the abstract
match = re.split(
    r"\\begin\s*{\s*abstract\s*}|\\s*\\section\s*{\s*Abstract\s*}",
    content,
    maxsplit=1,
    flags=re.IGNORECASE
)
if len(match) > 1:
    #return match[0].strip()
    #res_list.append(match[0].strip())
    #yield match[0].strip()
    pass

# If still not found, return the first 1/3 of the content as a fallback
content_length = len(content)
if content_length > 0:
    one_third_length = max(content_length//3, 2000)
    #return content[:one_third_length].strip()
    #res_list.append(content[:one_third_length].strip())
    #yield content[:one_third_length].strip()
    pass

# If still not found, return an empty string
#if res_list:
#  yield res_list
#else:
# yield ["",]



In [179]:
latex_extracted_institutions

['\\author{\n    Sonish Sivarajkumar, MS\\textsuperscript{1}\\textsuperscript{,2}\\thanks{Present address: School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA. Work was done while at Molecular Robotics, Kerala, India.},\n    Pratyush Tandale, MS\\textsuperscript{3}, \n    Ankit Bhardwaj, BS\\textsuperscript{4}, \\\\\n    Kipp W. Johnson, MD,PhD\\textsuperscript{5}, \n    Anoop Titus, MD\\textsuperscript{6}, \n    Benjamin S. Glicksberg, PhD\\textsuperscript{7},\\\\\n    Shameer Khader, PhD, MPH\\textsuperscript{8}\\textsuperscript{\\dag}, \n    Kamlesh K. Yadav, PhD\\textsuperscript{9, 10}\\textsuperscript{\\dag}, \\\\\n    Lakshminarayanan Subramanian, PhD\\textsuperscript{4}\\thanks{Corresponding authors: shameer.khader20@imperial.ac.uk, kamlesh.yadav@tamu.edu, lakshmi@cs.nyu.edu}\n}',
 ',\n    Pratyush Tandale, MS',
 ',\n    Pratyush Tandale, MS',
 ', \n    Ankit Bhardwaj, BS',
 ', ',
 ', \n    Anoop Titus, MD',
 ', \n    Benjamin S. Glicksberg, PhD'

In [177]:
def extract_texsuperscript(latex_node_list, res=None):
    '''for each superscript, get the contents of the next LatexCharsNode'''
    bailout_macros = set(['abstract', 'subsection'])
    if res is None:
        res = []
    for i, node in enumerate(latex_node_list):
        sublist = []
        #print(type(node))
        try:
            if node.macroname=='textsuperscript':
                run_started = False
                text_list = []
                for nnode in latex_node_list[i:]:
                    #print(type(nnode))
                    if isinstance(nnode, LatexCharsNode):
                        text_list.append(nnode.latex_verbatim())
                        run_started = True
                    elif isinstance(nnode, LatexMacroNode):
                        if nnode.macroname == '&':
                            text_list.append('&')
                        elif run_started:
                            break
                    elif not isinstance(nnode, LatexCharsNode):
                        if run_started:
                            break
                if text_list:
                    res.append(" ".join(text_list))
        except AttributeError:
            pass
        if isinstance(node, LatexMacroNode):
            try: 
                if node.macroname in bailout_macros:
                    break
                sublist = node.nodeargd.argnlist
                #print(sublist)
            except AttributeError:
                pass
        if isinstance(node, (LatexGroupNode, LatexEnvironmentNode)):
            try:
                sublist = node.nodelist
                #print(sublist)
            except AttributeError:
                pass
        if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document':
            break
        if sublist:
            extract_texsuperscript(sublist, res)
    return res

In [162]:
for node in nodelist[139].nodeargd.argnlist[0].nodelist:
    print(node)
    print('\n')
    print('--')
    print('\n')

LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2703, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')


--


LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2719, len=6, nodelist=[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2720, len=4, macroname='dag', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')], delimiters=('{', '}'))


--


LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2725, len=21, chars='Corresponding authors')


--




In [163]:
extract_texsuperscript(nodelist[139:140])  #nodelist[138:150])

['Corresponding authors']

[LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2702, len=45, nodelist=[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2703, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2719, len=6, nodelist=[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2720, len=4, macroname='dag', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')], delimiters=('{', '}')), LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2725, len=21, chars='Corresponding authors')], delimiters=('{', '}'))]

In [167]:
extract_texsuperscript(nodelist[139:])

['Corresponding authors',
 'Molecular Robotics, Kerala, India; ',
 'School of Computing and Information, University of Pittsburgh, Pennsylvania, PA, USA; ',
 'Health Informatics ',
 'Department of Computer Science, Courant Institute of Mathematical Sciences, New York University, New York, NY, USA; ',
 'Institute for Next Generation Healthcare, Mount Sinai Health System, New York, NY, USA; ',
 'Department of Preventive Cardiology, DeBakey Heart ',
 'Hasso Plattner Institute for Digital Health, Icahn School of Medicine at Mount Sinai, New York, NY, USA; ',
 'Faculty of Medicine, Imperial College London, London, UK; ',
 'School of Engineering Medicine,  Texas A',
 'Department of Translational Medical Sciences, Center for Genomic and Precision Medicine, Texas A']

In [103]:
extra_pats = ["\\textsuperscript"]

[pat in l for l in latex_extracted_institutions for pat in extra_pats]

[True]

In [115]:
focus_nodes

[(137,
  LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=1902, len=794, macroname='author', nodeargd=ParsedMacroArgs(argspec='{', argnlist=[LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=1909, len=787, nodelist=[LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=1910, len=28, chars='\n    Sonish Sivarajkumar, MS'), LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=1938, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=1954, len=3, nodelist=[LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=1955, len=1, chars='1')], delimiters=('{', '}')), LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=1957, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexGroupNode(parsing_state=<parsing state 140674525854880>, 

In [111]:
nodelist[141].nodeargd.argnlist[0].nodelist

[LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2756, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexGroupNode(parsing_state=<parsing state 140674525854880>, pos=2772, len=3, nodelist=[LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2773, len=1, chars='1')], delimiters=('{', '}')),
 LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2775, len=35, chars='Molecular Robotics, Kerala, India; '),
 LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2810, len=2, macroname='\n', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexCharsNode(parsing_state=<parsing state 140674525854880>, pos=2812, len=6, chars='      '),
 LatexMacroNode(parsing_state=<parsing state 140674525854880>, pos=2818, len=16, macroname='textsuperscript', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexGroupNode(parsing_state=<pars

## Test

In [ ]:
os.cpu_count()

In [ ]:
# importlib.reload(phase_one)

## times

```
Batch size    parallel workers    thread workers    time               n       sec/item        errors
   5             2                   5                                 100     
  10             2                   5                                 100     
  10             2                  10                                 100     
  
   5             3                   5                2 min 50s        100     1.70
  10             3                   5                2 min  8s        100     1.28
  10             3                  10                2 min 23s        100     1.43
  
  10             8                  10                3min 44s        1000      .22              0
  20             8                  10                3min 15s.       1000      .21              0       6414, 255; 6717, 296
  20             8                  20                3min 27s        1000      .21              0
  50             8                  25                3min 21s        1000      .20 
  
  25            12                  25                8min 1s         1000      .48
 100            12                  25                12min 39s       1000      .73 
 

10                   3                          5                    1 min 14s
```

In [ ]:
%%time

test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv")
#test_ids_df.head()
ids_2311_all = test_ids_df["arx_id"].unique()


tt = TicToc()

input_ids = set(ids_2311_all)
save_name = "2311_db"
sample_size = "all"
batch_size = 20
parallel_workers = 8
thread_workers = 10

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    return res

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        for future in as_completed(futures): #tqdm(as_completed(futures), total=len(futures)):
            res = future.result()
            res_list.extend(res)
#        res = [future.result() for future in concurrent.futures.as_completed(futures)]
#        for batch in res:
#            res_list.extend(batch)
    return res_list

def format_results(arxid_inst_ror_list):
    res_list = []
    for key, group in tqdm(itr.groupby(arxid_inst_ror_list, key=lambda x: x[0])):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = 'null'
            if len(x) == 3:
                ror = x[2].strip()
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list



try:
    objects = []
    with open(f"checkpoints/{save_name}_{sample_size}.pkl", 'rb') as cp_fp:
        while True:
            try:
                obj = pickle.load(cp_fp)
                objects.append(obj)
            except EOFError:
                break
except FileNotFoundError as e:
    pass

known_ids = []
known_res = []
for obj in objects:
    known_ids.extend(x[0] for x in obj)
    known_res.extend(obj)

known_ids = set(known_ids)
input_ids = input_ids - known_ids
print(f"Found checkpoints for {len(known_ids)} nodes.")

if sample_size != "all":
    input_ids = input_ids[:sample_size]

#import concurrent.futures
#import phase_one


os.environ["TOKENIZERS_PARALLELISM"] = "false" 

batches = np.array_split(list(input_ids), len(input_ids)//batch_size)
tt.tic()
print(f"Start: {len(input_ids)} in {len(batches)} batches")
with open(f"checkpoints/{save_name}_{sample_size}.pkl", 'ab') as cp_fp:
    res = run_phase_one_in_parallel(batches, cp_fp)
tt.toc()
known_res.extend(res)
res_df = pd.DataFrame.from_records(known_res, columns=['arx_id', 'name', 'ror'])
res_df.to_csv(f"gs://institutional-extract-scratch/output/{save_name}_{sample_size}.csv.zip", index=False)
#res_list = format_results(res)
tt.toc()

In [ ]:
len(res)
sum( 1 for x in res if x[1] == 'error' )
sum( 1 for x in res if x[2] == 'null' )

In [ ]:
len(res)
sum( 1 for x in res if x[1] == 'error' )
sum( 1 for x in res if x[2] == 'null' )

In [ ]:
res_df = pd.DataFrame.from_records(res, columns=['arx_id', 'name', 'ror'])
res_df.to_csv(f"gs://institutional-extract-scratch/output/2311_scopus_{sample_size}.csv.zip", index=False)

In [ ]:
res

In [ ]:
%%time

tt = TicToc()

sample_size = 100
batch_size = 20
parallel_workers = 3
thread_workers = 20
#import concurrent.futures
#import phase_one

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    return res

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        res = [future.result() for future in concurrent.futures.as_completed(futures)]
        for batch in res:
            res_list.extend(batch)
    return res_list

def run_phase_two_in_sequence(arxid_inst_list):
    res_list = []
    for key, group in tqdm(itr.groupby(arxid_inst_list, key=lambda x: x[0])):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = get_ror(clean_name)
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list

os.environ["TOKENIZERS_PARALLELISM"] = "false" 
input_ids = scopus_all[:sample_size]
batches = np.array_split(input_ids, len(input_ids)//batch_size)
tt.tic()
print(f"Start Phase 1, {len(input_ids)} in {len(batches)} batches")
res = run_phase_one_in_parallel(batches)
tt.toc()
res[:5]
print("Start Phase 2")
res_list = run_phase_two_in_sequence(res)
tt.toc()
res_list[:5]

In [ ]:
%%time

tt = TicToc()

sample_size = 100
batch_size = 20
parallel_workers = 3
thread_workers = 20
#import concurrent.futures
#import phase_one

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list, max_workers=thread_workers)
    print(len(res))
    res_list = run_phase_two_in_sequence(res)
    return res_list

def run_phase_one_in_parallel(arx_id_batches):
    res_list = []
    total_len = sum(len(x) for x in arx_id_batches)
    with concurrent.futures.ProcessPoolExecutor(max_workers=parallel_workers) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        res = [future.result() for future in concurrent.futures.as_completed(futures)]
        for batch in res:
            res_list.extend(batch)
    return res_list

def run_phase_two_in_sequence(arxid_inst_list):
    res_list = []
    for key, group in itr.groupby(arxid_inst_list, key=lambda x: x[0]):
        ror_inst = []
        for x in group:
            clean_name = x[1].split('.', 1)[-1].strip()
            ror = get_ror(clean_name)
            inst = {
                'name':clean_name,
                'ror_id':ror
            }
            ror_inst.append(inst)
        arx_rec = {
            "arxiv_id": key.strip(),
            "institutions_with_ror": ror_inst,
        }
        res_list.append(arx_rec)
    return res_list


os.environ["TOKENIZERS_PARALLELISM"] = "false" 
input_ids = scopus_all[:sample_size]
batches = np.array_split(input_ids, len(input_ids)//batch_size)
tt.tic()
print(f"Start Phase 1, {len(input_ids)} in {len(batches)} batches")
res = run_phase_one_in_parallel(batches)
tt.toc()
res[:5]
#print("Start Phase 2")
#res_list = run_phase_two_in_sequence(res)
#res_list[:5]

In [ ]:
res[:5]

In [ ]:
res_list

## Scratch